# Within-segment repeated-structure validation

Test whether a DPT-free count-Gaussian atlas orders structures **within** the supplied PT-S1/S2/S3 labels, and whether that ordering predicts held-out genes in the other healthy control specimen. The outer validation is Ctrl1A2 → Ctrl1A4 and the reverse; each direction is repeated across three deterministic held-out gene folds. Profile rows are repeated cross-sections, not independent mice.

The privileged segment-plus-coverage oracle is diagnostic only: it receives query segment labels and three metadata covariates, while the atlas and soft-segment baseline use expression features only. Five within-specimen × segment position permutations test fine order beyond coarse segment labels. No physical depth or DPT is required or used. All-gene library exposure retains the compositional dependence between modeled and held-out counts.

In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.joint_repeated_atlas import fit_joint_atlas,project_joint_atlas
from pseudospace.repeated_structure import fit_atlas
from pseudospace.repeated_atlas_baselines import fit_pc1_reference,project_pc1_reference,fit_segment_mixture,predict_segment_mixture
from pseudospace.within_segment_validation import (fit_segment_covariate_oracle,predict_segment_covariate_oracle,permute_positions_within_segments)
from pseudospace.stage_cache import cached_payload
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'within_segment_repeated_structure_validation'
OUT.mkdir(parents=True,exist_ok=True)
CONTROLS=('Ctrl1A2','Ctrl1A4')
NOTEBOOK_LOGIC_VERSION='18.within_segment.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
NULL_SEEDS=(11,23,37,51,79)
GRID=np.linspace(0,1,101)
MODEL_PARAMS=dict(grid=GRID,roughness=.001,offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','joint_repeated_atlas.py','repeated_structure.py','repeated_atlas_baselines.py','within_segment_validation.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,components=N_COMPONENTS,folds=N_FOLDS,null_seeds=NULL_SEEDS,markers=sorted(MARKERS),model={**MODEL_PARAMS,'grid':GRID.tolist()},gene_spline_ridge=.001,oracle_ridge=.01,classifier_C=1.,selected_per_segment=50,crossfit_hash='sha256(structure_id)%2')

## Healthy inputs and fixed partitions

Use the same healthy controls, marker exclusions, and three deterministic gene folds as notebook 17. Here each specimen serves as the entire outer training or query specimen. Labels enter the weak ordinal atlas fit, privileged oracle, and evaluation only.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_used_for_fitting=False,depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

## Projection, prediction, and scoring helpers

Every fitted preprocessing transform, atlas, classifier, and response spline is trained on the source specimen only. A held-out specimen receives zero Gaussian offset. The primary score is query-gene MSE normalized gene-wise by the equal-specimen training variance (floor 1e-3), summarized for all query structures and within each segment.

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def orient_pca(Y,a):
    value=Y[:,0].copy()
    if np.median(value[a==2])<np.median(value[a==0]): value=-value
    lo,hi=np.quantile(value,[.01,.99])
    if hi<=lo: raise ValueError('Degenerate training PC1 initializer')
    return np.clip((value-lo)/(hi-lo),0,1)

def project_gaussian(atlas,Y,offset):
    return project_joint_atlas(atlas,Y,offset=offset)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0) for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,pred in predictions.items():
        scaled=(response-pred)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any(): out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),normalized_mse=float(scaled[use].mean())))
    return out

## Six outer transfers and training-only selected panels

For each source specimen and each held-out gene fold, train the count representation on non-marker genes outside that fold, then fit the weakly anchored joint atlas. The query projection uses expression only. For supplementary gene selection, split the source structures by a fixed structure-ID hash; compare cross-half actual-coordinate decoding against a segment-plus-coverage covariate oracle. Rank each segment’s genes by positive held-out normalized-MSE gain and retain at most 50. Query genes never determine this selection.

In [ ]:
def run_validation():
    score_rows_out, selected_rows, diagnostics, failures = [], [], [], []
    for source in CONTROLS:
        target = next(s for s in CONTROLS if s != source)
        tr, te = specimens == source, specimens == target
        for fold in range(N_FOLDS):
            key = dict(source_specimen=source, test_specimen=target, fold=fold)
            try:
                mg = np.flatnonzero(~marker_mask & (gene_fold != fold))
                eg = np.flatnonzero(~marker_mask & (gene_fold == fold))
                if len(mg) < 2 or len(eg) < 1:
                    raise ValueError('Insufficient modeling or held-out genes')
                train_counts = dense_panel(raw_counts, tr, mg)
                query_counts = dense_panel(raw_counts, te, mg)
                Y, transform = fit_count_representation(
                    train_counts, raw_total[tr], specimens[tr], N_COMPONENTS,
                    theta=100., seed=SEED)
                query_Y = transform_count_representation(
                    query_counts, raw_total[te], transform)
                atlas = fit_joint_atlas(
                    Y, specimens[tr], anatomy[tr], orient_pca(Y, anatomy[tr]),
                    **MODEL_PARAMS)
                if not atlas['converged']:
                    raise RuntimeError('Count-Gaussian atlas did not converge')
                train_z = project_gaussian(
                    atlas, Y, atlas['offsets'][source])['z_mean']
                query_z = project_gaussian(
                    atlas, query_Y, np.zeros(Y.shape[1]))['z_mean']

                train_response = X[tr][:, eg]
                query_response = X[te][:, eg]
                train_sample = specimens[tr]
                variance = normalized_variance(train_response, train_sample)
                continuous = fit_gene_spline(
                    train_response, train_z, train_sample).predict(query_z)

                pc = fit_pc1_reference(Y, anatomy[tr])
                pc_train = project_pc1_reference(pc, Y)
                pc_query = project_pc1_reference(pc, query_Y)
                pc_prediction = fit_gene_spline(
                    train_response, pc_train, train_sample).predict(pc_query)
                mixture = fit_segment_mixture(
                    Y, train_response, anatomy[tr], train_sample,
                    C=1., seed=SEED)
                soft_prediction = predict_segment_mixture(
                    mixture, query_Y)['prediction']

                train_detected = np.asarray((raw_counts[tr][:, mg] > 0).sum(axis=1)).ravel()
                query_detected = np.asarray((raw_counts[te][:, mg] > 0).sum(axis=1)).ravel()
                train_cov = np.column_stack([
                    np.log(raw_total[tr]), np.log1p(n_spots[tr]),
                    np.log1p(train_detected)])
                query_cov = np.column_stack([
                    np.log(raw_total[te]), np.log1p(n_spots[te]),
                    np.log1p(query_detected)])
                oracle = fit_segment_covariate_oracle(
                    train_response, train_cov, anatomy[tr], train_sample,
                    ridge=.01)
                oracle_prediction = predict_segment_covariate_oracle(
                    oracle, query_cov, anatomy[te])

                predictions = {
                    'actual_coordinate': continuous,
                    'count_PC1': pc_prediction,
                    'soft_segment': soft_prediction,
                    'segment_coverage_oracle': oracle_prediction,
                }
                for null_seed in NULL_SEEDS:
                    null_z = permute_positions_within_segments(
                        train_z, train_sample, anatomy[tr], null_seed)
                    null_fit = fit_gene_spline(
                        train_response, null_z, train_sample)
                    predictions[f'within_segment_null_{null_seed}'] = null_fit.predict(query_z)

                meta = {**key, 'n_model_genes': len(mg), 'n_heldout_genes': len(eg)}
                score_rows_out.extend(primary_rows(
                    query_response, predictions, variance, anatomy[te], meta))

                # Cross-fit the held-out-gene decoder in both fixed hash halves.
                half = np.array([
                    int(hashlib.sha256(s.encode()).hexdigest()[:8], 16) % 2
                    for s in ids[tr]])
                actual_sq, oracle_sq, cv_anatomy = [], [], []
                for fit_half, eval_half in [(0, 1), (1, 0)]:
                    fit_use, eval_use = half == fit_half, half == eval_half
                    if fit_use.sum() < 4 or eval_use.sum() < 2:
                        raise ValueError('Insufficient training cross-fit half support')
                    response_fit, response_eval = train_response[fit_use], train_response[eval_use]
                    sample_fit = train_sample[fit_use]
                    zfit, z_eval = train_z[fit_use], train_z[eval_use]
                    spline = fit_gene_spline(response_fit, zfit, sample_fit)
                    actual_sq.append((response_eval - spline.predict(z_eval)) ** 2)
                    cv_anatomy.append(anatomy[tr][eval_use])
                    covmodel = fit_segment_covariate_oracle(
                        response_fit, train_cov[fit_use], anatomy[tr][fit_use],
                        sample_fit, ridge=.01)
                    oracle_prediction_cv = predict_segment_covariate_oracle(
                        covmodel, train_cov[eval_use], anatomy[tr][eval_use])
                    oracle_sq.append((response_eval - oracle_prediction_cv) ** 2)
                actual_cv = np.concatenate(actual_sq)
                oracle_cv = np.concatenate(oracle_sq)
                cv_anatomy = np.concatenate(cv_anatomy)

                selected_by_segment = {}
                for seg, segname in enumerate(['S1', 'S2', 'S3']):
                    cv_rows = cv_anatomy == seg
                    train_rows = anatomy[tr] == seg
                    scale = np.maximum(np.var(train_response[train_rows], axis=0), 1e-3)
                    gain = (oracle_cv[cv_rows].mean(axis=0)
                            - actual_cv[cv_rows].mean(axis=0)) / scale
                    eligible = np.flatnonzero(gain > 0)
                    order = eligible[np.argsort(-gain[eligible], kind='stable')][:50]
                    selected_by_segment[seg] = order
                    query_segment = anatomy[te] == seg
                    for j in order:
                        row = {
                            **key, 'segment': segname, 'gene': genes[eg[j]],
                            'training_cv_gain': float(gain[j]),
                        }
                        for method, prediction in predictions.items():
                            if query_segment.any():
                                error = np.mean((query_response[query_segment, j]
                                                 - prediction[query_segment, j]) ** 2) / scale[j]
                            else:
                                error = np.nan
                            row[f'{method}_query_gene_error'] = float(error)
                        row['null_mean_query_gene_error'] = float(np.mean([
                            row[f'within_segment_null_{seed}_query_gene_error']
                            for seed in NULL_SEEDS]))
                        selected_rows.append(row)

                    if len(order) and query_segment.any():
                        panel_vars = scale[order]
                        for method, prediction in predictions.items():
                            err = ((query_response[query_segment][:, order]
                                    - prediction[query_segment][:, order]) ** 2) / panel_vars
                            score_rows_out.append({
                                **meta, 'method': method, 'segment': segname + '_selected',
                                'n': int(query_segment.sum()),
                                'normalized_mse': float(err.mean()),
                                'selected_gene_count': int(len(order)),
                            })

                diagnostics.append({
                    **meta, 'atlas_converged': bool(atlas['converged']),
                    'iterations': len(atlas['history']), 'sigma2': float(atlas['sigma2']),
                    'model_genes': len(mg), 'heldout_genes': len(eg),
                    'positive_selected': {k: len(v) for k, v in selected_by_segment.items()},
                })
                print('Completed transfer:', source, '→', target, 'fold', fold, flush=True)
            except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                failures.append({**key, 'error': str(exc)})
                print('Transfer failure:', failures[-1], flush=True)

    frames = {
        'scores': pd.DataFrame(score_rows_out),
        'selected_genes': pd.DataFrame(selected_rows),
        'diagnostics': pd.DataFrame(diagnostics),
        'failures': pd.DataFrame(failures),
    }
    return {name: frame.to_json(orient='table') for name, frame in frames.items()}

In [ ]:
CACHE_INPUTS = dict(
    X=X, counts=raw_counts, library=raw_total, ids=ids, specimens=specimens,
    labels=labels, n_spots=n_spots, gene_fold=gene_fold, markers=marker_mask,
)
payload = cached_payload(
    'within_segment_repeated_structure_validation', run_validation,
    root=OUT/'stage_cache', params=CACHE_PARAMS,
    inputs=CACHE_INPUTS, code=CODE_DIGEST)
outputs = {
    name: pd.read_json(io.StringIO(str(value)), orient='table')
    for name, value in payload.items()
}
for name, frame in outputs.items():
    frame.to_csv(OUT/(name + '.csv'), index=False)
display(outputs['scores'].query("segment == 'all'").pivot_table(
    index=['source_specimen', 'test_specimen', 'fold'],
    columns='method', values='normalized_mse'))
display(outputs['scores'].query("segment in ['S1','S2','S3']").pivot_table(
    index=['source_specimen', 'test_specimen', 'fold', 'segment'],
    columns='method', values='normalized_mse'))
display(outputs['failures'])
if (not outputs['failures'].empty
        or len(outputs['diagnostics']) != len(CONTROLS) * N_FOLDS):
    raise RuntimeError(
        f'Incomplete six-fit outer validation; see {OUT}/failures.csv and diagnostics.csv')


## Reading this validation

The all-gene score is primary; segment-specific scores test whether performance persists within coarse labels. The selected-gene panel is supplementary and selected entirely inside the source specimen by cross-fitting. The segment-plus-coverage oracle receives privileged query labels and metadata; it diagnoses how much a coverage program can predict but is not a spatial-atlas competitor. The permutation null preserves specimen and segment membership while destroying fine within-segment coordinate assignment.

These profile-level comparisons describe transfer between two sections. They do not create independent biological replication, provide p-values, establish physical nephron distance, or remove the marker-derived information inherited through coarse labels. Coverage covariates can reflect both measurement depth and real anatomy.

## Distinguish whole-transcriptome prediction from transferred programs

The selected panels contain at most 50 response genes per training segment and gene fold, chosen solely by the training cross-fit. They are a supplementary test, not a replacement for the all-gene endpoint. Points below denote three rotated gene folds; they are not independent biological replicates or confidence intervals. No panel selection changes after seeing query errors.

In [ ]:
import matplotlib.pyplot as plt
summary=outputs['scores'].pivot_table(index=['source_specimen','test_specimen','fold','segment'],columns='method',values='normalized_mse')
null_columns=[name for name in summary.columns if name.startswith('within_segment_null_')]
summary['null_mean']=summary[null_columns].mean(axis=1)
summary['gain_vs_null_percent']=100*(1-summary.actual_coordinate/summary.null_mean)
summary['gain_vs_coverage_percent']=100*(1-summary.actual_coordinate/summary.segment_coverage_oracle)
summary.to_csv(OUT/'prediction_gain_summary.csv')
display(summary[['gain_vs_null_percent','gain_vs_coverage_percent']].groupby(['source_specimen','segment']).mean())
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,suffix,title in zip(axes,['','_selected'],['All withheld genes','Training-selected gene programs']):
    for offset,source,color in [(-.18,'Ctrl1A2','#4477AA'),(.18,'Ctrl1A4','#CC6677')]:
        target=next(s for s in CONTROLS if s!=source)
        for i,segment in enumerate(['S1','S2','S3']):
            values=summary.xs((source,target),level=('source_specimen','test_specimen')).xs(segment+suffix,level='segment')['gain_vs_coverage_percent'].to_numpy()
            ax.bar(i+offset,values.mean(),width=.33,color=color,alpha=.7,label=source+' → '+target if i==0 else None)
            ax.scatter(i+offset+np.linspace(-.06,.06,len(values)),values,s=18,color=color,zorder=3)
    ax.axhline(0,color='.5',linewidth=.8)
    ax.set(xticks=[0,1,2],xticklabels=['S1','S2','S3'],title=title,ylabel='Atlas MSE reduction vs segment+coverage oracle (%)')
    ax.spines[['top','right']].set_visible(False)
axes[0].legend(frameon=False,fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'within_segment_program_prediction.png',dpi=180)
plt.show()
# Overlap is descriptive: both selected lists used their own training specimen.
selected_overlap=[]
for segment in ['S1','S2','S3']:
    first=set(outputs['selected_genes'].query("source_specimen=='Ctrl1A2' and segment==@segment").gene)
    second=set(outputs['selected_genes'].query("source_specimen=='Ctrl1A4' and segment==@segment").gene)
    selected_overlap.append(dict(segment=segment,first_count=len(first),second_count=len(second),overlap=len(first&second)))
display(pd.DataFrame(selected_overlap))

## Nonlinear coverage oracle on the frozen gene panels

The previous oracle was linear in three coverage covariates. Here we add squares and pairwise interactions after training-only, within-segment standardization. This diagnostic remains privileged: it receives query segment labels, while the atlas does not. The original training-selected gene panels stay fixed; no response genes are reselected from query results. Coverage can include real morphology, so this is a competing predictor rather than proof of a technical artifact. Neither degree nor ridge is selected on query outcomes.

In [ ]:
QUADRATIC_VERSION='18.quadratic_coverage.1'
QUADRATIC_PARAMS=dict(version=QUADRATIC_VERSION,degree=2,ridge=.01,
    frozen_panels='original_training_crossfit_positive_top50',
    covariates=['log_allgene_library','log1p_n_spots','log1p_detected_model_genes'])

def run_quadratic_oracle():
    rows_out=[]; gene_rows=[]; diagnostics=[]; failures=[]
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr=specimens==source; te=specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold))
                eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                train_response=X[tr][:,eg]; query_response=X[te][:,eg]
                train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel())])
                query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel())])
                model=fit_segment_covariate_oracle(train_response,train_cov,anatomy[tr],specimens[tr],ridge=.01,degree=2)
                pred=predict_segment_covariate_oracle(model,query_cov,anatomy[te])
                meta={**key,'n_model_genes':len(mg),'n_heldout_genes':len(eg)}
                rows_out.extend(primary_rows(query_response,{'quadratic_segment_coverage_oracle':pred},normalized_variance(train_response,specimens[tr]),anatomy[te],meta))
                panel=outputs['selected_genes']
                panel=panel[(panel.source_specimen==source)&(panel.fold==fold)]
                gene_index={g:j for j,g in enumerate(genes[eg])}
                for seg,segname in enumerate(['S1','S2','S3']):
                    selected=panel[panel.segment==segname]
                    if selected.gene.duplicated().any():
                        raise ValueError('Repeated selected response gene')
                    if selected.empty:
                        continue
                    jj=np.array([gene_index[g] for g in selected.gene],int)
                    use=anatomy[te]==seg
                    scale=np.maximum(np.var(train_response[anatomy[tr]==seg],axis=0),1e-3)
                    errors=np.mean((query_response[use][:,jj]-pred[use][:,jj])**2,axis=0)/scale[jj]
                    rows_out.append(dict(**meta,method='quadratic_segment_coverage_oracle',segment=segname+'_selected',n=int(use.sum()),normalized_mse=float(errors.mean()),selected_gene_count=len(jj)))
                    for gene,error in zip(selected.gene,errors):
                        gene_rows.append(dict(**key,segment=segname,gene=gene,quadratic_oracle_query_gene_error=float(error)))
                outside=[]
                for seg in (0,1,2):
                    ref=train_cov[anatomy[tr]==seg]
                    query=query_cov[anatomy[te]==seg]
                    lo,hi=np.quantile(ref,[.01,.99],axis=0)
                    outside.append(float(np.mean(np.any((query<lo)|(query>hi),axis=1))))
                diagnostics.append(dict(**key,degree=model['degree'],query_outside_training_covariate_1_99_by_segment=outside))
                print('Quadratic coverage oracle:',source,'→',target,'fold',fold,flush=True)
            except (ValueError,RuntimeError,np.linalg.LinAlgError,KeyError) as exc:
                failures.append(dict(**key,error=str(exc)))
    frames={'scores':pd.DataFrame(rows_out),'selected_genes':pd.DataFrame(gene_rows),'diagnostics':pd.DataFrame(diagnostics),'failures':pd.DataFrame(failures)}
    return {name:frame.to_json(orient='table') for name,frame in frames.items()}

quadratic_payload=cached_payload('quadratic_coverage_oracle',run_quadratic_oracle,
    root=OUT/'stage_cache',params=QUADRATIC_PARAMS,
    inputs={**CACHE_INPUTS,'frozen_selected_genes':outputs['selected_genes']},code=CODE_DIGEST)
quadratic_outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in quadratic_payload.items()}
for name,frame in quadratic_outputs.items():
    frame.to_csv(OUT/('quadratic_oracle_'+name+'.csv'),index=False)
display(quadratic_outputs['failures'])
if not quadratic_outputs['failures'].empty or len(quadratic_outputs['diagnostics'])!=len(CONTROLS)*N_FOLDS:
    raise RuntimeError('Incomplete nonlinear coverage comparison')
comparison=pd.concat([outputs['scores'],quadratic_outputs['scores']],ignore_index=True)
comparison=comparison.pivot_table(index=['source_specimen','test_specimen','fold','segment'],columns='method',values='normalized_mse')
comparison['gain_vs_quadratic_percent']=100*(1-comparison.actual_coordinate/comparison.quadratic_segment_coverage_oracle)
comparison.to_csv(OUT/'quadratic_oracle_comparison.csv')
display(comparison[['actual_coordinate','segment_coverage_oracle','quadratic_segment_coverage_oracle','gain_vs_quadratic_percent']].groupby(['source_specimen','segment']).mean())
paired_genes=outputs['selected_genes'].merge(quadratic_outputs['selected_genes'],on=['source_specimen','test_specimen','fold','segment','gene'],validate='one_to_one')
if len(paired_genes)!=len(outputs['selected_genes']):
    raise ValueError('Frozen selected gene panel changed in nonlinear comparison')
paired_genes.to_csv(OUT/'quadratic_oracle_paired_genes.csv',index=False)
display(quadratic_outputs['diagnostics'])

## Reusable estimator equivalence check

Fit the public count-based interface on Ctrl1A2, using the original fold-0 modeling mask and parameters, then project Ctrl1A4 from counts and exposure alone. Verify that the Gaussian coordinates reproduce the original withheld-gene benchmark. The paired NB projection remains a separate observation-model estimate; disagreement and conditional entropy are diagnostics, not calibrated uncertainty or proof of anatomy. This is an implementation equivalence check, not an additional biological replicate.

In [ ]:
from pseudospace.pseudostructure_atlas import fit_pseudostructure_atlas,project_pseudostructure_atlas
PUBLIC_API_VERSION='18.public_api.1'
PUBLIC_API_DIGEST=hashlib.sha256(CODE_DIGEST.encode()+b''.join(
    (REPO/'pseudospace'/name).read_bytes() for name in
    ['pseudostructure_atlas.py','count_rate_atlas.py'])).hexdigest()
def run_public_api_check():
    source,target,fold=CONTROLS[0],CONTROLS[1],0
    tr,te=specimens==source,specimens==target
    mask=~marker_mask & (gene_fold!=fold)
    eg=np.flatnonzero(~marker_mask & (gene_fold==fold))
    reference=fit_pseudostructure_atlas(
        raw_counts[tr],raw_total[tr],specimens[tr],anatomy[tr],genes,mask,
        n_components=N_COMPONENTS,seed=SEED,**MODEL_PARAMS)
    projection=project_pseudostructure_atlas(
        reference,raw_counts[te],raw_total[te],genes,ids[te])
    response=X[tr][:,eg]
    prediction=fit_gene_spline(response,reference['training_z'],specimens[tr]).predict(
        projection['gaussian']['z_mean'])
    scores=pd.DataFrame(primary_rows(
        X[te][:,eg],{'public_api_gaussian':prediction},
        normalized_variance(response,specimens[tr]),anatomy[te],
        dict(source_specimen=source,test_specimen=target,fold=fold)))
    positions=pd.DataFrame(dict(structure_id=projection['structure_ids'],
        gaussian_z=projection['gaussian']['z_mean'],count_z=projection['count']['z_mean'],
        gaussian_entropy=projection['gaussian']['entropy'],
        count_entropy=projection['count']['entropy'],
        model_disagreement=projection['coordinate_disagreement']))
    return dict(scores=scores.to_json(orient='table'),
        positions=positions.to_json(orient='table'),
        converged=bool(reference['gaussian']['converged']))
public_payload=cached_payload('public_pseudostructure_api_equivalence',run_public_api_check,
    root=OUT/'stage_cache',params=dict(version=PUBLIC_API_VERSION,source=CONTROLS[0],fold=0,
        components=N_COMPONENTS,seed=SEED,model={**MODEL_PARAMS,'grid':GRID.tolist()},
        count_theta=100.,rate_bandwidth=.08,min_effective=12.),
    inputs=CACHE_INPUTS,code=PUBLIC_API_DIGEST)
public_scores=pd.read_json(io.StringIO(public_payload['scores']),orient='table')
public_positions=pd.read_json(io.StringIO(public_payload['positions']),orient='table')
public_scores.to_csv(OUT/'public_api_scores.csv',index=False)
public_positions.to_csv(OUT/'public_api_positions.csv',index=False)
# Validation runs on cache hits too; never cache the guard itself.
original=outputs['scores'].query("source_specimen == @CONTROLS[0] and fold == 0 and method == 'actual_coordinate'")
check=public_scores.merge(original[['segment','normalized_mse']],on='segment',
    suffixes=('_public','_original'),validate='one_to_one')
if not public_payload['converged'] or len(check)!=4:
    raise RuntimeError('Incomplete public API equivalence check')
np.testing.assert_allclose(check.normalized_mse_public,check.normalized_mse_original,
    rtol=1e-8,atol=1e-10)
display(check)
display(public_positions[['gaussian_entropy','count_entropy','model_disagreement']].describe())
print('Public API reproduces the original four Gaussian benchmark scores.')